# Exp6 — Pooled-Metric Reanalysis
Recomputes the manuscript-matched pooled endpoint-MSE gain from saved Exp6 row-level results.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path('results_highdim_robustness')
EXPECTED_POOLED_GAIN = {('Electricity',96):-0.0445,('Electricity',192):1.4992,('Electricity',336):-2.1513,('Electricity',720):0.6136,('PeMS04',12):-0.5522,('PeMS04',24):-3.0977,('PeMS04',48):28.9576}


In [ ]:
files = sorted(ROOT.rglob('*_rows.csv'))
frames=[]
needed={'dataset','target_strategy','target_seed','candidate_cap','target_channel','horizon','shared_test_mse','adaptive_test_mse','adaptive_gain_vs_shared_%'}
for p in files:
    df=pd.read_csv(p)
    if needed.issubset(df.columns): frames.append(df)
if not frames: raise FileNotFoundError('Run the full Exp6 notebook first.')
all_rows=pd.concat(frames,ignore_index=True)


## Manuscript-matched pooled metric
For each condition, the primary gain is `100 * (sum Shared MSE - sum Adaptive MSE) / sum Shared MSE`.


In [ ]:
def pooled_gain(g):
    s=float(g['shared_test_mse'].sum()); a=float(g['adaptive_test_mse'].sum())
    return 100.0*(s-a)/max(s,1e-12)
keys=['dataset','target_strategy','target_seed','candidate_cap','horizon']
rows=[]
for key,g in all_rows.groupby(keys):
    d,strategy,seed,cap,h=key
    rows.append({'dataset':d,'target_strategy':strategy,'target_seed':int(seed),'candidate_cap':int(cap),'horizon':int(h),'n_targets':int(g['target_channel'].nunique()),'pooled_gain_pct':pooled_gain(g),'mean_targetwise_gain_pct':float(g['adaptive_gain_vs_shared_%'].mean())})
pooled=pd.DataFrame(rows).sort_values(keys).reset_index(drop=True)
display(pooled.round(4))


## Original-protocol reproduction and robustness summaries


In [ ]:
orig=pooled[(pooled.target_strategy=='even') & (pooled.candidate_cap==128)].copy()
orig['expected']=[EXPECTED_POOLED_GAIN.get((d,int(h)),np.nan) for d,h in zip(orig.dataset,orig.horizon)]
orig['abs_difference_pctpt']=np.abs(orig.pooled_gain_pct-orig.expected)
display(orig.round(4))
rand=pooled[(pooled.target_strategy=='random') & (pooled.candidate_cap==128)]
subset=rand.groupby(['dataset','horizon'],as_index=False).agg(n_subset_seeds=('target_seed','nunique'),mean_pooled_gain_pct=('pooled_gain_pct','mean'),std_pooled_gain_pct=('pooled_gain_pct','std'),min_pooled_gain_pct=('pooled_gain_pct','min'),max_pooled_gain_pct=('pooled_gain_pct','max'),positive_seed_fraction=('pooled_gain_pct',lambda s: float((np.asarray(s)>0).mean())))
display(subset.round(4))
caps=pooled[pooled.target_strategy=='even'].pivot_table(index=['dataset','horizon'],columns='candidate_cap',values='pooled_gain_pct').reset_index()
display(caps.round(4))
